In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 


In [2]:
df = pd.read_csv(r"C:\Users\MSI-1\Downloads\cardekho_imputated.csv")

In [3]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [4]:

df.drop('brand',axis = 1,inplace = True)

In [5]:
num_features = [features for features in df.columns if df[features].dtype != 'o']
cat_features = [features for features in df.columns if df[features].dtype == 'o']
discrete_features = [features for features in num_features if len(df[features].unique() )<= 25]
continuous_features=[feature for feature in num_features if feature not in discrete_features]

In [6]:
from sklearn.model_selection import train_test_split
X = df.drop(['car_name','selling_price','Unnamed: 0'],axis = 1)
y = df['selling_price']

In [7]:
y

0         120000
1         550000
2         215000
3         226000
4         570000
          ...   
15406     250000
15407     925000
15408     425000
15409    1225000
15410    1200000
Name: selling_price, Length: 15411, dtype: int64

In [8]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [9]:
X['model'] = le.fit_transform(X['model'])

In [10]:
X

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5
...,...,...,...,...,...,...,...,...,...,...
15406,117,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
15407,42,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
15408,77,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
15409,114,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7


In [11]:
num_features = X.select_dtypes(exclude='object').columns
ca_features = ['seller_type','fuel_type','transmission_type']

from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

ohe = OneHotEncoder()
std = StandardScaler()
preprocessor = ColumnTransformer(

    [
        ('onehotencoder',ohe,ca_features),
        ('standard',std,num_features)

    ],remainder = 'passthrough'
)

In [12]:
X = preprocessor.fit_transform(X)

In [13]:
X

array([[ 0.        ,  1.        ,  0.        , ..., -1.32425883,
        -1.26335238, -0.40302241],
       [ 0.        ,  1.        ,  0.        , ..., -0.55471774,
        -0.43257082, -0.40302241],
       [ 0.        ,  1.        ,  0.        , ..., -0.55471774,
        -0.47911321, -0.40302241],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.02291783,
         0.06822523, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ...,  1.32979434,
         0.91715831,  2.07344426],
       [ 1.        ,  0.        ,  0.        , ...,  0.02099878,
         0.39588361, -0.40302241]], shape=(15411, 17))

In [14]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [18]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
}

In [19]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [20]:
for i in range(len(list(models))):
    reg = list(models.values())[i]
    reg.fit(X_train,y_train)

     # Make predictions
    y_train_pred = reg.predict(X_train)
    y_test_pred = reg.predict(X_test)
    
    # Evaluate Train and Test dataset
    
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

    

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 559313.7144
- Mean Absolute Error: 268437.6549
- R2 Score: 0.6183
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 507556.7988
- Mean Absolute Error: 281329.9042
- R2 Score: 0.6575


Lasso
Model performance for Training set
- Root Mean Squared Error: 559313.7153
- Mean Absolute Error: 268436.2654
- R2 Score: 0.6183
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 507556.0066
- Mean Absolute Error: 281328.0471
- R2 Score: 0.6575


Ridge
Model performance for Training set
- Root Mean Squared Error: 559314.5848
- Mean Absolute Error: 268394.1969
- R2 Score: 0.6183
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 507539.6755
- Mean Absolute Error: 281279.8973
- R2 Score: 0.6575


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 338800.6151
- Mean 

In [21]:
#Initialize few parameter for Hyperparamter tuning
knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

In [22]:
# Models list for Hyperparameter tuning
randomcv_models = [('KNN', KNeighborsRegressor(), knn_params),
                   ("RF", RandomForestRegressor(), rf_params)
                   
                   ]

In [23]:
##Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

c:\Users\MSI-1\anaconda3\envs\munich\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 6 is smaller than n_iter=100. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits


c:\Users\MSI-1\anaconda3\envs\munich\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
90 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
56 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\MSI-1\anaconda3\envs\munich\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\MSI-1\anaconda3\envs\munich\Lib\site-packages\sklearn\base.py", line 1329, in wrapper
    estimator._validate_params()
  File "c:\Users\MSI-1\anaconda3\envs\munich\Lib\site-packages\sklearn\base.py", line 492, in _validate_params
    validate_parameter_constraints(
  

---------------- Best Params for KNN -------------------
{'n_neighbors': 2}
---------------- Best Params for RF -------------------
{'n_estimators': 1000, 'min_samples_split': 2, 'max_features': 7, 'max_depth': None}


In [25]:
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, min_samples_split=2, max_features=7, max_depth=None, 
                                                     n_jobs=-1),
     "K-Neighbors Regressor": KNeighborsRegressor(n_neighbors=10, n_jobs=-1)
    
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)

Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 142682.6781
- Mean Absolute Error: 39820.4273
- R2 Score: 0.9752
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 228316.9985
- Mean Absolute Error: 101445.3091
- R2 Score: 0.9307
K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 376091.6943
- Mean Absolute Error: 105661.5556
- R2 Score: 0.8274
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 300966.1939
- Mean Absolute Error: 123243.5391
- R2 Score: 0.8796
